# 第5回：提出から運用・監視・再学習（MLOps）へ

この回は3つのパートで構成します：**Kaggleに入って最初の提出を作る ／ Kaggle改善会 ／ Show & Tellと自社データへの橋渡し**。

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

**AIと一緒に進める**：分からないコードは、セル全体ではなく気になる数行をM365 CopilotなどのAIへ貼って
説明や修正を相談します（`ASK COPILOT`）。ただしAIの答えは鵜呑みにせず、必ず自分の出力で確かめます。

`TRY`は全員、`CHANGE`は値を1つ変える練習、`CHALLENGE`は余裕がある人向け、
`DEEP DIVE`・`APPENDIX`は発展です（飛ばしても本編は完結します）。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

模擬コンペで提出と改善を体験し、最後にモデルを「作って終わり」にせず、運用・監視・再学習のループ（MLOps）へつなげます。

- 問題・指標・データ・提出形式を読み解き、再現可能なベースラインを作る
- cross_val_predictでOOF予測を作り、CVとLBの一致を確かめる
- 提出CSVを検査する関数を、テスト付きで書く
- 限られた時間で実験を優先順位付けし、OOFスタッキングで統合する
- adversarial validationで学習とテストの分布ずれを点検する
- 複数シードの平均と閾値調整で、偶然に頼らない改善を積む
- モデルの目的・検証・結果・限界を短く説明し、再現可能に共有する
- 学習済みPipelineをjoblibで保存し、モデルカードを関数で生成する
- 適用領域と較正の観点から、使ってよい範囲と監視項目を決める

### この回の進み方（大切）

この回は、旧カリキュラムの**3回分をまとめた長い回**です。**パート1→2→3**の順に、各パートの
`CORE`（本線）で手を動かします。1回の時間で全部を終える必要はありません。各パートの
`DEEP DIVE`／`APPENDIX`は、余裕のある人や自習で進めてください。日をまたいで少しずつでも大丈夫です。

### 先に押さえる言葉

- Leaderboard：提出結果を順位表示する仕組み
- OOF予測：交差検証の検証側だけを集めた予測
- submission：指定形式の予測ファイル
- CV-LBギャップ：手元の検証と公開スコアの差
- ベースライン：最初に必ず保存する比較起点
- OOFスタッキング：OOF予測を入力に上位モデルで統合する方法
- 分布ずれ：学習とテストで入力の分布が違うこと
- シードアンサンブル：乱数だけ変えた複数モデルの平均
- 閾値調整：確率からクラスへの境界を変えること
- 実験統合：有効な変更を再検証しながら組み合わせること
- モデルカード：用途・データ・評価・限界をまとめた記録
- 適用領域：モデルを使ってよい対象と条件
- 永続化：学習済みモデルをファイルへ保存すること
- ドリフト：運用後に入力や関係が変わること
- 監視：運用後の入力や性能変化を確認すること

> **実行前の30秒予想**：各パートの問いに、今の言葉で仮の答えを書いてから始めます。


---

# パート1：Kaggleに入って最初の提出を作る

**このパートの問い：コンペの説明を、ローカルの分析手順へどう翻訳するか。**


## コンペは「これまでの総合演習」

Kaggle（や、この教材のローカル模擬コンペ）は、第1〜12回で学んだことを1本の流れにする総合演習です。
新しい手法は増えません。大事なのは、**コンペの説明を、いつもの分析手順へ翻訳する**こと。

最初に必ず4点を確認します：**目的（何を予測）／評価指標／データ（trainとtestの違い）／提出形式**。
まずデータを開いて形を見ます。


In [ ]:
import pandas as pd
train = pd.read_csv(DATA / "local_competition" / "train.csv")
test = pd.read_csv(DATA / "local_competition" / "test.csv")
sample = pd.read_csv(DATA / "local_competition" / "sample_submission.csv")
print("train:", train.shape, "test:", test.shape, "提出見本:", sample.shape)
display(train.head(3))
display(sample.head(3))


### 出力の読み方

- **trainには`active`列があり、testには無い**はずです。testの答えは伏せられていて、提出して初めて採点されます。
- **提出見本(sample_submission)**は「こういう形で出してね」という雛形。列名と行数を必ずこれに合わせます。
- trainとtestの行数を足すと、第3回で見た元データの件数に対応します。


## コンペ説明（この模擬コンペの4点）

- **目的**：実験計画時の情報から活性`active`（0/1）を予測する
- **指標**：F1（第8回。活性が少ないのでaccuracyでなくF1）
- **データ**：`train.csv`には答えあり、`test.csv`には無し
- **提出形式**：`sample_id`と`active`の2列

Kaggle Titanicを使う場合も、最初にこの4点（目的・指標・train/test・提出形式）を同じように確認します。


## ベースラインを作る（第9回のPipelineを再利用）

第9回で学んだ`ColumnTransformer`＋`Pipeline`をそのまま使い、数値もカテゴリも安全に1つのモデルへ通します。
`sample_id`や実験後の列など、**使ってはいけない列を`drop_columns`で外す**のがポイント（第5回のリーク回避）。
まずローカルの検証F1で当たりを付けます。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

target = "active"
# batch_id（実験バッチの通し番号）とexperiment_dateは、活性とは無関係な「ID的な列」。
# 特徴量に入れるとノイズになり、One-Hotで列だけ増えるので外す（第5回の予測時点の考え方）。
drop_columns = ["sample_id", "experiment_date", "batch_id", "smiles", target]
features = [column for column in train.columns if column not in drop_columns]
numeric = train[features].select_dtypes(include="number").columns.tolist()
categorical = [column for column in features if column not in numeric]
preprocess = ColumnTransformer([
    ("数値", SimpleImputer(strategy="median"), numeric),
    ("カテゴリ", Pipeline([("補完", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])
model = Pipeline([("前処理", preprocess), ("モデル", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42))])
X_train, X_valid, y_train, y_valid = train_test_split(train[features], train[target], test_size=0.25, random_state=42, stratify=train[target])
model.fit(X_train, y_train)
print("ローカル検証F1:", round(f1_score(y_valid, model.predict(X_valid)), 3))


### 出力の読み方と、`scaffold_group`の注意

このローカル検証F1が、あなたの**最初のものさし**です。以降の改善は、必ずこの値と比べます。
「提出して順位が上がったか」だけでなく、**手元の検証がどう動いたか**を先に見る習慣が、コンペで
崩れないコツです（次のDEEP DIVEのCV-LBの話につながります）。

補足：ここでは`scaffold_group`（化合物系列）をカテゴリ特徴量として使っていますが、第6回のとおり本来は
**系列を跨がない分割（GroupKFold）とセットで扱うべき列**です。乱数分割のまま使うと評価が楽観的に
なり得ます。余力があれば、この列を外す・またはGroup分割にすると手元スコアがどう変わるか試してください。


## TRY：提出CSVを作り、機械的に検査する

提出でいちばん多い失敗は、モデルの精度ではなく**フォーマットのミス**（列名・行数・余計なindex列）。
`assert`で自動チェックしてから保存します。`index=False`で余計な行番号列を混ぜないことも重要です。


In [ ]:
model.fit(train[features], train[target])
submission = pd.DataFrame({"sample_id": test["sample_id"], "active": model.predict(test[features])})
assert list(submission.columns) == ["sample_id", "active"]
assert len(submission) == len(test)
assert submission["sample_id"].is_unique
output = ROOT / "workspace" / "submission_baseline.csv"
submission.to_csv(output, index=False)
print("保存先:", output)
submission.head()


### 出力の読み方

3つの`assert`を通ってCSVが保存されれば、形式は合格。`workspace/`に出力されるので、Kaggleが使える人は
これをアップロードします。使えない場合は、講師がローカルで採点します（第14回）。

## CHANGE

提出前に変えるのは**1点だけ**（第12回の原則）。例：`max_depth=6`を`3`へ変え、ローカル検証F1がどう動くか
確認してから提出します。


## DEEP DIVE：手元でLeaderboardを予想する（OOF）と、提出を守る

コンペで沼にはまる典型が「提出回数を無駄遣いして、手元で何も分かっていない」状態です。
**OOF予測**で手元にLeaderboard相当の推定を持ち、**提出バリデータ**で形式ミスを防ぎます。


### OOF予測：提出せずにスコアを見積もる

`cross_val_predict`は、各行を「その行を学習に使っていないモデル」で予測します（OOF＝out-of-fold）。
これを全部集めれば、**提出しなくても**手元でLeaderboardに近いF1を推定できます。提出回数の節約になります。


In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import f1_score

oof = cross_val_predict(model, train[features], train[target], cv=StratifiedKFold(5, shuffle=True, random_state=42))
print("OOF F1:", round(f1_score(train[target], oof), 3))
print("この値は、公開スコアの当たりを付ける手元の推定として使える。")


### 出力の読み方

このOOF F1と、実際に提出したときのスコア（LB）を比べます。**両者が近ければ**手元の検証は信頼でき、
改善の判断を手元だけで進められます。**大きく食い違えば**、分布ずれ（第6回のadversarial validation）や
リークを疑います。この差を**CV-LBギャップ**と呼びます。


### 提出バリデータを「テスト」する

第2回で学んだ「テストで守る」を提出に適用します。検査関数を書くだけでなく、**わざと壊した提出**を
渡して、すべての`assert`がちゃんと弾くかを確かめます。関数が本当に機能する保証になります。


In [ ]:
def validate_submission(sub, test, expected=("sample_id", "active")):
    "提出CSVの列・行数・ID一致・値域を検査する。問題があればAssertionError。"
    expected = list(expected)
    assert list(sub.columns) == expected, "列名または順序が違います"
    assert len(sub) == len(test), "行数がtestと一致しません"
    assert sub[expected[0]].is_unique, "IDが重複しています"
    assert sub[expected[0]].tolist() == test[expected[0]].tolist(), "IDの順序がtestと一致しません"
    assert sub[expected[1]].isin([0, 1]).all(), "予測値は0/1にしてください"
    return "提出形式OK"

print(validate_submission(submission, test))
broken = submission.copy()
broken.loc[broken.index[0], "active"] = 5
try:
    validate_submission(broken, test)
except AssertionError as error:
    print("異常を検出:", error)


### 出力の読み方

正しい提出は「提出形式OK」を返し、`active`に5を混ぜた壊れた提出は「異常を検出: 予測値は0/1に…」で
弾かれます。**弾かれることを確認して初めて**、検査関数は信頼できます。本番の提出前に必ず通す関数として
手元に残しておきましょう。


## APPENDIX（任意・追加演習）

提出づくりを効率化します。90分の外の自習向けです。まず**どんなモデルでも提出CSVを作る関数**を用意し、
複数モデルの提出を量産します。


In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

def make_submission(estimator, name):
    "モデルを全trainで学習し、testを予測して提出CSVを保存する。"
    est = clone(estimator).fit(train[features], train[target])
    sub = pd.DataFrame({"sample_id": test["sample_id"], "active": est.predict(test[features])})
    path = ROOT / "workspace" / f"submission_{name}.csv"
    sub.to_csv(path, index=False)
    print(f"{name}: 保存 {path.name}  陽性率={sub['active'].mean():.3f}")
    return sub

make_submission(model, "rf")
_ = make_submission(Pipeline([("前処理", preprocess), ("モデル", LogisticRegression(max_iter=1000))]), "logit")


### 出力の読み方

2つの提出が`workspace/`に保存されます。**陽性率**（活性と予測した割合）がモデル間で大きく違うなら、
判定の癖が違うということ。`clone`で毎回まっさらなモデルから学習しているので、状態の混ざりがありません。


### どの特徴量が効いているか（提出モデルの中身）

提出に使ったRandom Forestの特徴量重要度を、Pipelineの中から取り出します。One-Hot後の列名で表示されます。


In [ ]:
fitted = model.named_steps["モデル"]
names = model.named_steps["前処理"].get_feature_names_out()
imp = pd.DataFrame({"特徴量": names, "重要度": fitted.feature_importances_}).sort_values("重要度", ascending=False)
display(imp.head(10).round(3))


### 出力の読み方

上位の特徴量が、モデルが判定に使っている主な手がかりです。第1・12回のとおり不純度重要度は偏りが
あるので、余裕があれば並べ替え重要度でも確認します。化学的に納得できる列が上位なら、ひとまず安心です。


### OOFとholdout、2つの手元推定を比べる

提出せずに性能を見積もる方法は複数あります。**OOF**（第13回本編）と、単純な**holdout**（1回の取り置き）を
比べ、両者が近いかを確認します。近ければ手元の検証は安定しています。


In [ ]:
from sklearn.model_selection import cross_val_predict, train_test_split, StratifiedKFold
from sklearn.metrics import f1_score

oof = cross_val_predict(clone(model), train[features], train[target], cv=StratifiedKFold(5, shuffle=True, random_state=42))
Xh_tr, Xh_va, yh_tr, yh_va = train_test_split(train[features], train[target], test_size=0.25, random_state=0, stratify=train[target])
hold = clone(model).fit(Xh_tr, yh_tr)
print("OOF F1     :", round(f1_score(train[target], oof), 3))
print("holdout F1 :", round(f1_score(yh_va, hold.predict(Xh_va)), 3))


### 出力の読み方

2つが近ければ、手元の推定は信頼できます。OOFは全データを検証に使えるぶん安定しやすく、holdoutは
1回きりなので振れやすい。**複数の見積もりが一致するか**を確認する習慣が、コンペでも実務でも効きます。


---

# パート2：Kaggle改善会

**このパートの問い：限られた時間で、次に何を試すか。**


## 改善会：限られた時間で「次の一手」を選ぶ

ベースラインができたら、次は改善です。ただし時間は有限。**闇雲に試すのではなく、分担して1人1変更**を
検証し、良かったものだけを統合します。ここでも第12回の原則（1度に1つ、同じ条件、記録を残す）が効きます。

いちばん大事な心得：**手元の検証（ローカル）とLeaderboardの両方を見る**こと。Leaderboardだけを追うと、
公開スコアに過剰適合して最終順位を落とします。まず、答え合わせ用の`answers`も含めてデータを読みます。


In [ ]:
import pandas as pd
train = pd.read_csv(DATA / "local_competition" / "train.csv")
test = pd.read_csv(DATA / "local_competition" / "test.csv")
answers = pd.read_csv(DATA / "local_competition" / "instructor_answers.csv")


## 5人の担当

1人1テーマに分かれます：**1. 欠損補完 / 2. 特徴量（最適温度からの距離）/ 3. モデルの深さ /
4. 判定閾値 / 5. 誤分類の確認**。全員が同じ`random_state=42`とF1を使い、**担当箇所以外は変えない**——
こうすると「誰の変更が効いたか」を後で切り分けられます。


## 改善案を1つ組んで、ローカルで検証する

この例では2〜3の担当（特徴量追加＋浅い木＋`class_weight`）を1つの案にまとめています。第11回の
`temperature_distance`を足し、第8回の`class_weight="balanced"`で少数クラスを重視。まずローカル検証F1で
ベースラインと比べます。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

improved_train = train.copy()
improved_test = test.copy()
for frame in [improved_train, improved_test]:
    frame["temperature_distance"] = (frame["temperature_c"] - 78).abs()
target = "active"
# batch_id・experiment_dateは活性と無関係なID的な列なので特徴量から外す（第13回と同じ方針）
ignored = ["sample_id", "experiment_date", "batch_id", "smiles", target]
features = [c for c in improved_train.columns if c not in ignored]
numeric = improved_train[features].select_dtypes(include="number").columns.tolist()
categorical = [c for c in features if c not in numeric]
preprocess = ColumnTransformer([
    ("数値", SimpleImputer(strategy="median"), numeric),
    ("カテゴリ", Pipeline([("補完", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical),
])
model = Pipeline([("前処理", preprocess), ("モデル", RandomForestClassifier(n_estimators=300, max_depth=3, class_weight="balanced", random_state=42))])
X_train, X_valid, y_train, y_valid = train_test_split(improved_train[features], improved_train[target], test_size=0.25, random_state=42, stratify=improved_train[target])
model.fit(X_train, y_train)
print("改善案のローカルF1:", round(f1_score(y_valid, model.predict(X_valid)), 3))


### 出力の読み方

このローカルF1を、第13回のベースライン（`submission_baseline`を作ったときの検証F1）と比べます。
**上がっていれば採用候補**。ただし1回の分割なので、余裕があれば交差検証（第10回）で確かめると確実です。


## 模擬Leaderboardで答え合わせする

この教材では講師が`answers`（正解）を持っており、ローカルで「提出したつもり」の採点ができます。
全データで学習し直してtestを予測し、`answers`と突き合わせて**模擬Leaderboard F1**を出します。


In [ ]:
model.fit(improved_train[features], improved_train[target])
improved_submission = pd.DataFrame({"sample_id": improved_test["sample_id"], "active": model.predict(improved_test[features])})
merged = answers.merge(improved_submission, on="sample_id", suffixes=("_true", "_pred"))
print("模擬Leaderboard F1:", round(f1_score(merged["active_true"], merged["active_pred"]), 3))


### 出力の読み方と実験ログ

- **ローカルF1と模擬LB F1が近い**なら、手元の検証は信頼できます。**大きく食い違う**なら、過剰適合や分布ずれを疑います。
- 改善しても悪化しても、`変更点 / ローカルF1 / 模擬LB F1 / 気づき`を1行で記録します。
- **Leaderboardだけ上がってローカルが下がった案は要注意**（公開スコアへの過剰適合の疑い）。良い変更だけを慎重に統合します。


## DEEP DIVE：単体を超える3つの技

上位を狙うときの定番を3つ。**OOFスタッキング**（違うモデルを束ねる）、**分布ずれの点検**
（train/testが似ているか）、**シード平均**（乱数の偶然を薄める）です。いずれも第6・10回の応用です。


### OOFスタッキング：違うモデルの予測を束ねる

第10回のスタッキングを、コンペ流に手作りします。3つのモデルの**OOF確率**（第13回）を作り、それらを
入力にした上位モデル（ロジスティック回帰）で統合します。OOFを使うのは、束ねる段階でリークしないためです。


In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

pre = ColumnTransformer([
    ("n", SimpleImputer(strategy="median"), numeric),
    ("c", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", sparse_output=False)), categorical),
])
members = {
    "rf": make_pipeline(pre, RandomForestClassifier(n_estimators=300, max_depth=4, random_state=42)),
    "hgb": make_pipeline(pre, HistGradientBoostingClassifier(max_iter=200, random_state=42)),
    "logit": make_pipeline(pre, LogisticRegression(max_iter=1000)),
}
skf = StratifiedKFold(5, shuffle=True, random_state=42)
oof = {}
for name, est in members.items():
    oof[name] = cross_val_predict(est, improved_train[features], improved_train[target], cv=skf, method="predict_proba")[:, 1]
    print(f"{name:6s} OOF F1:", round(f1_score(improved_train[target], (oof[name] >= 0.5).astype(int)), 3))
meta_X = pd.DataFrame(oof)
stack_oof = cross_val_predict(LogisticRegression(max_iter=1000), meta_X, improved_train[target], cv=skf, method="predict_proba")[:, 1]
print("スタッキング OOF F1:", round(f1_score(improved_train[target], (stack_oof >= 0.5).astype(int)), 3))


### 出力の読み方

各モデル単体のOOF F1と、スタッキングのOOF F1を比べます。**スタッキングが単体最良を上回れば**束ねた
価値あり。ほぼ同じなら、モデルたちが似た間違え方をしている（束ねる旨みが少ない）ということ。第10回と
同じ教訓：束ねは万能ではありません。


### 分布ずれを点検する（adversarial validation）

第6回の手法をコンペに適用。trainとtestを見分ける分類器のAUCで、両者の分布の近さを測ります。
AUCが高ければ、ローカル検証がLeaderboardとずれる原因になります。


In [ ]:
from sklearn.model_selection import cross_val_score

combined = pd.concat([
    improved_train[numeric].assign(is_test=0),
    improved_test[numeric].assign(is_test=1),
], ignore_index=True)
adv = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, random_state=42))
auc = cross_val_score(adv, combined[numeric], combined["is_test"], cv=5, scoring="roc_auc")
print("adversarial validation AUC:", round(auc.mean(), 3), "（0.5付近なら分布は近い）")


### 出力の読み方

AUCが0.5付近なら、train/testは似ていて手元CVは信頼できます。高ければ、CV-LBギャップの一因。実データの
コンペでは、AUCを上げている列を特定して扱いを見直す、といった対処につなげます。


### シード平均：乱数の運を薄める

同じモデルでも`random_state`を変えると予測が少し変わります。複数シードの確率を平均すると、**乱数由来の
ばらつきが打ち消し合い**、安定した予測になります。少ない手間で効きやすい定番テクです。


In [ ]:
import numpy as np

probs = []
for seed in [0, 1, 2, 3, 4]:
    est = make_pipeline(pre, RandomForestClassifier(n_estimators=300, max_depth=4, random_state=seed)).fit(improved_train[features], improved_train[target])
    probs.append(est.predict_proba(improved_test[features])[:, 1])
ensemble_pred = (np.mean(probs, axis=0) >= 0.5).astype(int)
seed_merged = answers.merge(pd.DataFrame({"sample_id": improved_test["sample_id"], "active": ensemble_pred}), on="sample_id", suffixes=("_true", "_pred"))
print("5シード平均の模擬LB F1:", round(f1_score(seed_merged["active_true"], seed_merged["active_pred"]), 3))


### 出力の読み方

5シード平均の模擬LB F1が、単一シードのときより**わずかに高く・安定**していれば成功。派手さは
ありませんが、こうした地味で確実な積み上げが、コンペでも実務でも効きます。「1回の高スコア」より
「**再現できる改善**」を選ぶ——これが今日いちばん持ち帰ってほしい姿勢です。


## APPENDIX（任意・追加演習）

改善の詰めを、OOFを使って安全に行います（すべて上のDEEP DIVEで作った`oof`を再利用）。90分の外の
自習向けです。まず**2モデルの重み付き平均（ブレンド）**の最適な重みを、OOF上で探します。


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

y_true = improved_train[target]
best = None
for w in np.linspace(0, 1, 11):
    blend = w * oof["rf"] + (1 - w) * oof["hgb"]
    f1 = f1_score(y_true, (blend >= 0.5).astype(int))
    if best is None or f1 > best[1]:
        best = (round(w, 1), round(f1, 3))
print(f"rf重み={best[0]} のときOOF F1最大={best[1]}")


### 出力の読み方

重み0はhgbのみ、1はrfのみ、途中が混合。**単体より混合が良い重み**が見つかれば、ブレンドの価値あり。
OOFで重みを決めるのは、テストに触れずに（リークなく）調整するためです。ただし重みを探しすぎると
OOFに過剰適合するので、探索は粗め（ここは11点）にとどめます。


### 判定閾値もOOFで最適化する

第8回の閾値調整を、OOF確率に対して行います。0.5に固定せず、F1が最大になる閾値を手元で選びます。


In [ ]:
rows = []
for t in np.linspace(0.2, 0.8, 13):
    rows.append({"閾値": round(t, 2), "F1": f1_score(y_true, (oof["rf"] >= t).astype(int))})
tbl = pd.DataFrame(rows)
print("OOFでF1最大の閾値:", tbl.loc[tbl["F1"].idxmax(), "閾値"])
tbl.round(3)


### 出力の読み方

F1が最大になる閾値が0.5とずれるなら、閾値調整で無料の改善が得られます。**OOFで選んだ閾値を、最終提出に
だけ適用**します（検証に使った同じデータで選んで報告しない、という第6・12回の原則を守ります）。


### 誤分類を群別に分析する

どの化合物系列でよく間違えるかを、OOF予測で集計します。特定の系列に誤りが偏るなら、その系列を
表す特徴量の不足や、データ不足を疑います。次の改善仮説のきっかけになります。


In [ ]:
val = improved_train[["scaffold_group", target]].copy()
val["oof_pred"] = (oof["rf"] >= 0.5).astype(int)
val["誤り"] = val[target] != val["oof_pred"]
by_group = val.groupby("scaffold_group").agg(件数=("誤り", "size"), 誤り数=("誤り", "sum"), 誤り率=("誤り", "mean"))
display(by_group.sort_values("誤り率", ascending=False).round(3))


### 出力の読み方

誤り率の高い系列が、モデルの弱点。件数が十分あるのに誤り率が高い系列は、**その系列に効く特徴量を
足す**（第11回）か、**分割を系列単位にする**（第6回）といった次の一手につながります。エラー分析は、
闇雲なチューニングより効く改善のきっかけです。


---

# パート3：Show & Tellと自社データへの橋渡し

**このパートの問い：自社データで始めるなら、最初の小さな一歩は何か。**


## 最終回：成果を「伝え」、自社データへ「橋渡し」する

最後は、作ったものを人に伝え、次の一歩へつなぐ回です。データサイエンスは「良いモデルを作って終わり」
ではなく、**「使われて初めて価値になる」**。ここまで学んだことを、発表と持ち帰りの形にまとめます。

まず**再現性**の確認から。第14回のNotebookを`Kernel`→`Restart Kernel and Run All Cells`で頭から実行し、
同じ提出CSVができることを確かめます（第12回の再現性の実践）。次のセルは、発表で共有できる基本の数字を出します。


In [ ]:
import pandas as pd
experiment_data = pd.read_csv(DATA / "compound_experiments.csv")
print("共有する候補")
print("データ件数:", len(experiment_data))
print("活性率:", round(experiment_data["active"].mean(), 3))
print("収率の中央値:", experiment_data["yield_pct"].median())


### 読みどころ

こうした基本統計（件数・活性率・中央値）は、発表の最初に置くと聞き手が状況をつかめます。**派手な
モデルより、まずデータの素性を1〜2行で言える**ことが、信頼される発表の土台です。


## 1人5分のShow & Tell

次のうち1つを選んで共有します：**面白かった図 / 改善した実験 / 悪化したが学びがあった実験 /
Copilotへの良かった聞き方 / 自社テーマへ持ち帰りたい考え方**。

完成度は競いません。むしろ**「悪化したが学びがあった実験」**の共有が、チーム全体の学びになります
（うまくいかない筋を先に潰せる）。「1回の高スコア」より「再現できる気づき」を持ち寄ります。


## 自社テーマ1枚シート

この教材の集大成として、自分のテーマを1枚に落とします。機密情報や実データは書かず、一般化した
表現で。**第5回の問題設定がここに戻ってきます**——予測時点と使えない情報を、もう一度自分の言葉で。

| 項目 | 記入内容 |
|---|---|
| 利用者と判断 | 誰が何を決めるか |
| 予測時点 | いつ予測するか |
| 目的変数 | 何を予測するか |
| 説明変数候補 | その時点で得られる情報 |
| 使えない情報 | 未来情報、測定後情報、機密上使えない情報 |
| 評価方法 | 指標と分割単位 |
| 単純な基準 | 平均、最頻値、現在の判断方法など |
| 最初の実験 | 1〜2週間で試せる小さな範囲 |

この1枚が、勉強会後に自社データで踏み出す**最初の一歩の設計図**になります。


## DEEP DIVE：発表で終わらせない——再現・共有・安全な運用

発展として、実務で「モデルを渡す」ときに必要な3つを扱います。**永続化**（保存して再利用）、
**モデルカード**（使い方の説明書）、**適用領域**（予測してよい範囲）。どれも「モデルを安全に使ってもらう」
ための工夫です。


### 永続化：学習済みモデルをファイルに保存する

毎回学習し直すのは非効率で、再現性も損なわれます。`joblib`で学習済みPipelineを**丸ごと保存**し、
読み直しても**同じ予測**になることを`assert`で確かめます。前処理も一緒に保存される点が重要です。


In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

data = pd.read_csv(DATA / "compound_experiments.csv")
feat = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_tr, X_te, y_tr, y_te = train_test_split(data[feat], data["active"], test_size=0.25, random_state=42, stratify=data["active"])
final = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)).fit(X_tr, y_tr)
path = ROOT / "workspace" / "final_model.joblib"
joblib.dump(final, path)
reloaded = joblib.load(path)
assert np.array_equal(final.predict(X_te), reloaded.predict(X_te)), "保存前後で予測が一致しません"
print("保存し読み直しても同じ予測:", path)


### 読みどころ

`assert`が通り「同じ予測」と出れば、保存→配布→再利用の流れが安全に回ることの確認になります。
`Pipeline`ごと保存するので、**受け取った人は前処理を意識せず`predict`するだけ**。第9回でPipelineに
まとめた恩恵がここで効きます。


### モデルカード：使い方の説明書を関数で作る

モデルは「精度の数字」だけ渡してもトラブルの元です。**誰向けか・何を決めるためか・限界・禁止事項**を
1枚にまとめた**モデルカード**を、関数で自動生成します。第5回の問題設定が、そのまま説明書になります。


In [ ]:
from sklearn.metrics import f1_score

def build_model_card(name, estimator, X_valid, y_valid, notes) -> pd.DataFrame:
    "モデルの用途と評価をまとめた1枚のカードを作る。"
    pred = estimator.predict(X_valid)
    items = {
        "モデル名": name,
        "検証F1": round(f1_score(y_valid, pred), 3),
        "想定利用者": notes["利用者"],
        "支援する判断": notes["判断"],
        "既知の限界": notes["限界"],
        "使ってはいけない条件": notes["禁止"],
    }
    return pd.DataFrame({"項目": list(items), "内容": list(items.values())})

build_model_card("活性スクリーナ", reloaded, X_te, y_te, {
    "利用者": "実験担当者", "判断": "追試する候補の優先順位",
    "限界": "新規scaffoldでは精度低下の可能性", "禁止": "測定後の列を入力に使うこと",
})


### 読みどころ

出来上がったカードには、性能（F1）と**使う上での注意**が並びます。特に「使ってはいけない条件（測定後の
列を入力にしない）」は、第5〜6回のリークの教訓そのもの。**精度より先に限界を書く**のが、信頼される
モデル提供者の作法です。


### 適用領域：予測してよい範囲を数値化する

モデルは、学習データと似た試料には強いですが、かけ離れた試料では当てになりません。学習データからの
**近傍距離**を測り、遠すぎる（範囲外の）試料を「要確認」に自動で仕分けます。95%点を閾値にします。


In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

train_filled = X_tr.fillna(X_tr.median())
scaler = StandardScaler().fit(train_filled)
nn = NearestNeighbors(n_neighbors=5).fit(scaler.transform(train_filled))
train_dist = nn.kneighbors(scaler.transform(train_filled))[0].mean(axis=1)
threshold = np.quantile(train_dist, 0.95)
valid_dist = nn.kneighbors(scaler.transform(X_te.fillna(X_tr.median())))[0].mean(axis=1)
out_of_domain = valid_dist > threshold
print(f"適用領域外と判定された検証試料: {int(out_of_domain.sum())} / {len(valid_dist)} 件")
print("範囲外は予測を鵜呑みにせず、要確認に回す運用が考えられる。")


### 読みどころ、そして全15回のまとめ

範囲外と判定された試料は、予測を鵜呑みにせず人が確認する——これが**安全にAIを使う**ということです。

全15回を貫いた芯は1つ：**「良いスコア」ではなく「意味のある予測」**。予測時点を決め、ベースラインと比べ、
リークを避け、正しく評価し、1つずつ改善を記録し、限界とともに伝える。この習慣こそが、皆さんが自社
データへ持ち帰るいちばんの財産です。お疲れさまでした。


## APPENDIX（任意・追加演習）

「渡せる成果物」を実際に書き出します。90分の外の自習向けです。まず**モデルカードをMarkdown＋JSONで
保存**し、第三者が読める形にします。


In [ ]:
import json

card = build_model_card("活性スクリーナ", reloaded, X_te, y_te, {
    "利用者": "実験担当者", "判断": "追試候補の優先順位",
    "限界": "新規scaffoldで精度低下の可能性", "禁止": "測定後の列を入力に使うこと",
})
lines = ["# モデルカード", ""]
for _, r in card.iterrows():
    lines.append(f"- **{r['項目']}**: {r['内容']}")
(ROOT / "workspace" / "model_card.md").write_text("\n".join(lines), encoding="utf-8")

meta = {"features": feat, "n_train": int(len(X_tr)), "model": "RandomForest(max_depth=5)"}
(ROOT / "workspace" / "model_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("保存: workspace/model_card.md, workspace/model_meta.json")
print("\n".join(lines))


### 出力の読み方

`model_card.md`は人が読む説明書、`model_meta.json`は機械が読む来歴（使った特徴量・学習件数・モデル種別）。
モデルと一緒にこの2つを残すと、**半年後の自分や引き継ぎ先が再現・判断できます**。


### ドリフトを模擬する：入力がずれたら「監視」で気づけるか

運用後、測定装置のずれなどで入力分布が変わる（ドリフト）ことがあります。テストの温度を+20℃ずらし、
**正解ラベルが無くても異常に気づけるか**を確かめます。運用中は正解（活性の実測）がすぐには手に入らない
ため、F1のような指標は即座には測れません。だからこそ、正解なしで検知できる監視が重要になります。


In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

drift = X_te.copy()
drift["temperature_c"] = drift["temperature_c"] + 20

# (1) 正解が無くても分かる変化：予測の陽性率
print(f"予測の陽性率: {reloaded.predict(X_te).mean():.3f} → {reloaded.predict(drift).mean():.3f}")

# (2) 監視：元データとドリフト後を見分けられるか（adversarial validation, 第6回）
cols = X_te.columns.tolist()
combined = pd.concat([X_te.assign(is_drift=0), drift.assign(is_drift=1)], ignore_index=True)
filled = combined[cols].fillna(combined[cols].median())
auc = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42), filled, combined["is_drift"], cv=5, scoring="roc_auc").mean()
print(f"監視AUC: {auc:.3f}（0.5=変化なし / 1.0に近い=明確な分布変化）")

# 参考：正解が手に入ればF1でも確認できる（運用中は正解が遅れて届く）
print(f"参考F1: {f1_score(y_te, reloaded.predict(X_te)):.3f} → {f1_score(y_te, reloaded.predict(drift)):.3f}")


### 出力の読み方

- **監視AUCが0.5をはっきり上回る**なら、元データとドリフト後をモデルが見分けられる＝入力分布が変化した、という警報です。温度を+20℃ずらしたので、AUCは0.5より明確に高く出るはずです（1に近いほど変化が大きい）。
- **予測の陽性率**の変化も、正解ラベル無しで「何かが変わった」と気づける手がかりです。
- 一方、**参考F1は運用中すぐには測れません**（正解が遅れて届くため）。しかもこのデータ・特徴量では変化が小さく、性能指標だけに頼ると見逃しかねません。だからこそ、正解なしで異常を検知するadversarial validation（第6回）のような監視が実務で効きます。


### 成績表をファイルに書き出す

`classification_report`を表として保存します。発表資料や引き継ぎに添付できる、機械可読な成績表です。


In [ ]:
from sklearn.metrics import classification_report

rep = classification_report(y_te, reloaded.predict(X_te), target_names=["非活性", "活性"], output_dict=True)
rep_df = pd.DataFrame(rep).T.round(3)
rep_df.to_csv(ROOT / "workspace" / "classification_report.csv")
display(rep_df)


### 出力の読み方、そしてこの教材の終わりに

クラスごとのprecision/recall/F1と全体のaccuracyが表になり、CSVで保存されます。数字だけを渡すのではなく、
**モデルカード（用途と限界）＋メタ情報（来歴）＋成績表**をひとまとめに渡す——ここまでできれば、
「作って終わり」から「使ってもらえる」への橋を渡せています。全15回、おつかれさまでした。


---

# パート4：MLOpsの考え方（運用・監視・再学習）

ここまでで「良いモデルを作る」ことはできました。実務では、そこからが本番です。モデルは
**作って終わりではなく、動かし続けるループ**の中で価値を出します。

> **学習 → 提供（サービング）→ 監視 → 再学習 → …**

このループを回す考え方や道具をまとめて**MLOps**と呼びます。実はこの教材では、その部品を
すでに各所で触っています——`Pipeline`（再現性）、`joblib`での**永続化**、実験ログ（追跡）、
ドリフト監視、適用領域。ここではそれらを「運用のループ」として一本につなぎます。


### 提供（サービング）：学習済みモデルを「関数」として使えるようにする

運用では、新しい試料が来るたびに学習し直しません。**保存済みモデルを読み込み、予測だけを返す
関数**を用意します。前のパートで保存した`reloaded`（前処理ごと保存したPipeline）をそのまま使います。


In [ ]:
def predict_activity(samples):
    "新しい試料(DataFrame)へ、活性の予測(0/1)と確率を返す推論関数。"
    proba = reloaded.predict_proba(samples[feat])[:, 1]
    return pd.DataFrame(
        {"活性予測": (proba >= 0.5).astype(int), "活性確率": proba.round(3)},
        index=samples.index,
    )

display(predict_activity(X_te.head()))


### 出力の読み方

前処理ごと保存したPipelineなので、受け取った人は`predict_activity(新しいデータ)`を呼ぶだけで
予測できます。これが「サービング」の最小形です。Webサービスやバッチ処理も、裏でこの関数を
呼んでいるだけ、とイメージしてください。


### 監視と再学習：いつモデルを作り直すか

運用後は、次を定期的に見張ります（このパートまでで手を動かした道具が、そのまま使えます）。

- **入力のドリフト**：入力分布が学習時とずれていないか（第2回・このパートのadversarial validationの監視AUC）。正解が手に入らなくても検知できるのが利点。
- **予測の傾向**：予測の陽性率が急に変わっていないか。
- **性能**：正解ラベルが遅れて届いたら、F1などを計算し直す。
- **適用領域**：学習データから遠い入力が増えていないか（近傍距離）。

これらが目安を超えたら**再学習のトリガー**です。新しいデータを足して学習し直し、
**同じ検証（第2回）・同じ評価（このパート）で前のモデルと比較**してから入れ替えます。
作って終わりにせず、このループを回し続けることが、実データでモデルを役立て続けるコツです。


### 再現性チェックリスト（引き継ぎ・監査のために）

- データ生成・前処理・学習が**固定シードで再現**できる（この教材はすべてシード固定です）。
- モデルは**Pipelineごと保存**し、前処理を含めて復元できる。
- **モデルカード**（用途・限界・禁止条件）と**メタ情報**（使った特徴量・学習件数）を一緒に残す。
- 実験は**ログ**に残し、なぜその設定にしたかを後から説明できる。

ここまで来れば、「作って終わり」から「**運用でき、引き継げる**」モデルへの橋を渡せています。


---

## よくある誤り

- testの情報へ合わせて特徴量を決める
- 提出ファイルのindex列を混入させる
- 最初から公開Notebookを丸ごと写す
- 5人の変更を一度に統合する
- Leaderboardだけを目的関数にする
- 分布ずれを無視してランダム分割だけで判断する
- スコアだけを成果として示す
- 自社データの利用許可や来歴を省略する
- 本番投入を最初の試行にする

## SELF-STUDY（任意・30〜60分）

- OOFのF1と提出後スコアの差を記録し、原因を1つ推測する
- validate_submissionへ異常な提出を渡し、全assertが働くか試す
- 単体最良・投票・スタッキングのOOF F1を比較する
- adversarial validationのAUCが高い列を除いて再評価する
- 保存したPipelineを読み直し、同じ入力で同じ予測になるか検証する
- 適用領域スコアを閾値化し、範囲外の試料を要確認として仕分ける

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. OOF予測は何に使えるか
2. CV-LBギャップが大きいとき何を疑うか
3. 提出前に検査する項目は何か
4. OOFスタッキングの手順は何か
5. 分布ずれをどう検知するか
6. 改善を統合する順序はどうするか
7. このモデルは誰の何の判断を助けるか
8. 適用領域をどう数値化したか
9. 運用後に監視すべき指標は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
